In [1]:
import pandas as pd
import os
from zipfile import ZipFile
import requests

# ----------------------------
# 1. Download & Extract MovieLens 1M
# ----------------------------
dataset_url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
dataset_zip = "ml-1m.zip"
extracted_path = "ml-1m"

if not os.path.exists(extracted_path):
    print("Downloading MovieLens 1M dataset...")
    r = requests.get(dataset_url)
    with open(dataset_zip, "wb") as f:
        f.write(r.content)
    with ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Download and extraction done.")

# ----------------------------
# 2. Load all files
# ----------------------------
ratings_file = os.path.join(extracted_path, "ratings.dat")
movies_file = os.path.join(extracted_path, "movies.dat")
users_file  = os.path.join(extracted_path, "users.dat")

ratings = pd.read_csv(
    ratings_file, sep="::", engine="python",
    names=["UserID","MovieID","Rating","Timestamp"]
)
movies = pd.read_csv(
    movies_file, sep="::", engine="python",
    names=["MovieID","Title","Genres"], encoding="latin-1"
)
users = pd.read_csv(
    users_file, sep="::", engine="python",
    names=["UserID","Gender","Age","Occupation","Zip-code"]
)

# ----------------------------
# 3. Convert timestamp
# ----------------------------
ratings['Timestamp'] = pd.to_datetime(ratings['Timestamp'], unit='s')

# ----------------------------
# 4. Check for duplicates
# ----------------------------
print("Duplicate rows:")
print(f"Ratings: {ratings.duplicated().sum()}")
print(f"Movies:  {movies.duplicated().sum()}")
print(f"Users:   {users.duplicated().sum()}")

# Drop duplicates
ratings = ratings.drop_duplicates().reset_index(drop=True)
movies  = movies.drop_duplicates().reset_index(drop=True)
users   = users.drop_duplicates().reset_index(drop=True)

# ----------------------------
# 5. Check for missing values in detail
# ----------------------------
def report_missing(df, name):
    total_missing = df.isnull().sum().sum()
    if total_missing == 0:
        print(f"No missing values in {name}.")
    else:
        print(f"Missing values in {name}:")
        for col in df.columns:
            missing_count = df[col].isnull().sum()
            if missing_count > 0:
                print(f" - Column '{col}': {missing_count} missing")
        print("Rows with missing values:")
        print(df[df.isnull().any(axis=1)])

report_missing(ratings, "ratings")
report_missing(movies, "movies")
report_missing(users, "users")

# ----------------------------
# 6. Filter ratings to May → Dec 2000
# ----------------------------
start_date = '2000-05-01'
end_date   = '2000-12-31'
ratings = ratings[(ratings['Timestamp'] >= start_date) & (ratings['Timestamp'] <= end_date)].reset_index(drop=True)
print("\nRatings after filtering May → Dec 2000:", ratings.shape)


Download and extraction done.
Duplicate rows:
Ratings: 0
Movies:  0
Users:   0
No missing values in ratings.
No missing values in movies.
No missing values in users.

Ratings after filtering May → Dec 2000: (891189, 4)


In [2]:
# ----------------------------
# Create monthly slices
# ----------------------------
monthly_bins = pd.date_range(start=start_date, end='2001-01-01', freq='MS')
ratings['SliceNum'] = pd.cut(ratings['Timestamp'], bins=monthly_bins, right=False).cat.codes + 1

# ----------------------------
# Inspect each slice
# ----------------------------
slice_summary = []

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    num_ratings = slice_df.shape[0]
    num_movies  = slice_df['MovieID'].nunique()
    num_users   = slice_df['UserID'].nunique()
    slice_summary.append({
        'SliceNum': s,
        'Ratings': num_ratings,
        'UniqueMovies': num_movies,
        'UniqueUsers': num_users
    })

slice_summary_df = pd.DataFrame(slice_summary)
print(slice_summary_df)


   SliceNum  Ratings  UniqueMovies  UniqueUsers
0         1    67437          2920          486
1         2    54486          2913          508
2         3    90334          3135          778
3         4   182109          3298         1310
4         5    52421          3083          576
5         6    42294          2993          500
6         7   290793          3552         2357
7         8   111315          3331         1215


In [3]:
import os
import pandas as pd

# ----------------------------
# 1. Create folder for slice metadata
# ----------------------------
os.makedirs("slice_metadata", exist_ok=True)

# ----------------------------
# 2. Split movies into popularity tiers per slice
# ----------------------------
high_pct, mid_pct, low_pct = 0.2, 0.3, 0.5  # 20/30/50 split

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    movie_counts = slice_df.groupby('MovieID')['Rating'].count().sort_values(ascending=False)
    total_movies = len(movie_counts)
    
    # Determine cutoff indices
    high_cut  = int(total_movies * high_pct)
    mid_cut   = high_cut + int(total_movies * mid_pct)
    
    # Assign tiers
    movie_tiers = pd.DataFrame({'MovieID': movie_counts.index})
    movie_tiers['PopTier'] = 'low'  # default
    movie_tiers.loc[:high_cut-1, 'PopTier'] = 'high'
    movie_tiers.loc[high_cut:mid_cut-1, 'PopTier'] = 'medium'
    
    # Save slice metadata
    movie_tiers.to_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv", index=False)
    print(f"Slice {s}: {total_movies} movies → High:{high_cut}, Medium:{mid_cut-high_cut}, Low:{total_movies-mid_cut}")


Slice 1: 2920 movies → High:584, Medium:876, Low:1460
Slice 2: 2913 movies → High:582, Medium:873, Low:1458
Slice 3: 3135 movies → High:627, Medium:940, Low:1568
Slice 4: 3298 movies → High:659, Medium:989, Low:1650
Slice 5: 3083 movies → High:616, Medium:924, Low:1543
Slice 6: 2993 movies → High:598, Medium:897, Low:1498
Slice 7: 3552 movies → High:710, Medium:1065, Low:1777
Slice 8: 3331 movies → High:666, Medium:999, Low:1666


In [4]:
# ----------------------------
# User type assignment per slice
# ----------------------------
import os
import pandas as pd

slice_nums = sorted(ratings['SliceNum'].unique())
os.makedirs("slice_metadata", exist_ok=True)

threshold = 0.7  # HighProportion threshold to classify mainstream

for s in slice_nums:
    slice_df = ratings[ratings['SliceNum'] == s].copy()
    if slice_df.empty: 
        continue

    # Load movie popularity tiers for this slice
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv")
    movie_tier_map = dict(zip(movie_tiers.MovieID, movie_tiers.PopTier))

    # Map each rating to movie tier
    slice_df['Tier'] = slice_df['MovieID'].map(movie_tier_map)

    # Count number of ratings per tier per user
    tier_counts = slice_df.groupby(['UserID', 'Tier']).size().unstack(fill_value=0)
    # Ensure all columns exist
    for col in ['high', 'medium', 'low']:
        if col not in tier_counts.columns:
            tier_counts[col] = 0

    # Compute HighProportion
    tier_counts['Total'] = tier_counts['high'] + tier_counts['medium'] + tier_counts['low']
    tier_counts['HighProportion'] = tier_counts['high'] / tier_counts['Total']

    # Assign user type
    tier_counts['UserType'] = tier_counts['HighProportion'].apply(
        lambda x: 'mainstream' if x >= threshold else 'niche'
    )

    # Save result
    user_type_df = tier_counts[['UserType']].reset_index()
    user_type_df.to_csv(f"slice_metadata/user_labels_slice_{s}.csv", index=False)
    
    print(f"Slice {s}: Total Users = {len(user_type_df)}, Mainstream = {sum(user_type_df.UserType=='mainstream')}, Niche = {sum(user_type_df.UserType=='niche')}")


Slice 1: Total Users = 486, Mainstream = 233, Niche = 253
Slice 2: Total Users = 508, Mainstream = 239, Niche = 269
Slice 3: Total Users = 778, Mainstream = 332, Niche = 446
Slice 4: Total Users = 1310, Mainstream = 598, Niche = 712
Slice 5: Total Users = 576, Mainstream = 276, Niche = 300
Slice 6: Total Users = 500, Mainstream = 223, Niche = 277
Slice 7: Total Users = 2357, Mainstream = 1420, Niche = 937
Slice 8: Total Users = 1215, Mainstream = 596, Niche = 619


PROJECT README
Temporal Knowledge Graphs for Recommender Systems
================================================

1. What this project is about
-----------------------------

This project studies how user preferences and item popularity change over time,
and how recommender models can adapt to those changes instead of treating data as static.

Most recommender systems:
- Merge all interactions into one dataset
- Assume user behavior and item popularity are fixed
- Ignore temporal structure

That is unrealistic.

In this project, we:
- Split user–item interactions into time slices
- Build a separate Knowledge Graph (KG) for each slice
- Train models slice by slice
- Compare a baseline recommender with a more expressive final model

The goal is to evaluate whether modeling structure and time actually improves recommendations.


2. Why use a Knowledge Graph
----------------------------

A standard user–item matrix only captures:
“User A interacted with Movie B”.

A Knowledge Graph allows us to represent richer structure:
- Users have behavioral types (mainstream vs niche)
- Movies have genres
- Movies have popularity levels
- All of these can change over time

By encoding interactions as a graph:
- Simple graph models can be used as baselines
- Relation-aware models can exploit edge types
- The same data supports multiple model families


3. Why time slicing
-------------------

User behavior in May 2000 is not the same as in December 2000.

Instead of building one global graph, we:
- Split ratings into monthly time slices
- Build one KG per slice
- Allow users and movies to appear, disappear, or change context

This enables:
- Temporal evaluation
- Embedding warm-starting
- Studying preference drift


4. Models trained later
-----------------------

This preprocessing supports two categories of models:

1. Baseline model
   - Treats all edges as the same
   - Uses only graph structure
   - Ignores relation types

2. Final model
   - Uses relation types
   - Distinguishes different kinds of edges
   - Exploits richer KG structure

Both models use the same preprocessed data.


5. Dataset
----------

- MovieLens 1M
- Ratings filtered to May 2000 – December 2000
- Monthly time slices


6. Outputs per time slice
------------------------

For each slice, the following are created:
- Movie popularity tiers
- User behavioral types
- Slice-specific Knowledge Graph
- Node ID mappings for temporal alignment

All outputs are stored as CSV files.


7. Directory structure
----------------------

.
├── slice_metadata/
│   ├── movie_pop_tiers_slice_1.csv
│   ├── user_labels_slice_1.csv
│   ├── ...
│   └── kg/
│       ├── kg_edges_slice_1.csv
│       ├── user_map_slice_1.csv
│       ├── movie_map_slice_1.csv
│       └── ...


8. Movie popularity tiers
-------------------------

Files:
slice_metadata/movie_pop_tiers_slice_{s}.csv

Description:
- Movies are ranked by number of ratings within a slice
- Assigned to popularity tiers:
  - high
  - medium
  - low

These tiers capture short-term popularity, not global popularity.


9. User behavioral types
------------------------

Files:
slice_metadata/user_labels_slice_{s}.csv

Description:
- Users are labeled based on what they consume in that slice
- Users who mostly rate popular movies → mainstream
- Others → niche

User types can change across slices.


10. Knowledge Graphs
--------------------

Each slice has its own Knowledge Graph.


10.1 Edge list
--------------

Files:
slice_metadata/kg/kg_edges_slice_{s}.csv

Columns:
- Src     : source node ID
- Dst     : destination node ID
- RelType : relation type ID

All edges are stored as bidirectional.


10.2 Relation types
-------------------

Relation ID meanings:

0 → User ↔ Movie  
1 → Movie ↔ Genre  
2 → Movie ↔ Popularity Tier  
3 → User ↔ User Type  

Baseline models ignore RelType.
Final models use RelType.


11. Node IDs
------------

Node IDs are:
- Integers
- Unique within a slice
- Assigned in non-overlapping ranges by node type

This guarantees that users, movies, genres, tiers, and user types
cannot be confused with each other.


12. Node ID mappings (important)
--------------------------------

Files:
- slice_metadata/kg/user_map_slice_{s}.csv
- slice_metadata/kg/movie_map_slice_{s}.csv

These map original UserID / MovieID to slice-local node IDs.

They are required for:
- Matching entities across slices
- Warm-starting embeddings
- Temporal stabilization during training


13. What is NOT stored
----------------------

- No trained models
- No tensors
- No pickled Python objects
- No framework-specific formats

Everything is reconstructed from CSVs in training notebooks.


14. How a new training session should start
-------------------------------------------

A fresh session should:
1. Select a slice
2. Load KG edges and mappings
3. Convert data to tensors as needed
4. Train baseline and final models
5. Optionally align embeddings across slices


15. Summary
-----------

This project builds time-aware Knowledge Graphs from MovieLens data
to compare a baseline recommender against a relation-aware model
while explicitly modeling temporal dynamics.


K = 10

In [5]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import scipy.sparse as sp
import os

# ----------------------------
# 0. Deterministic Seeding
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ----------------------------
# 1. Metric Helper Functions
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.array(recommendation_counts)
    if np.sum(arr) == 0: return 0.0
    arr = np.sort(arr)
    n = len(arr)
    cumvals = np.cumsum(arr)
    gini = (n + 1 - 2 * np.sum(cumvals) / cumvals[-1]) / n
    return gini

def compute_metrics(model, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=10):
    model.eval()
    with torch.no_grad():
        # [MODIFIED]: Removed adj_matrix dependency
        final_user_emb, final_item_emb = model()
        
        valid_test = test_df[
            test_df['UserID'].isin(user_map.keys()) & 
            test_df['MovieID'].isin(item_map.keys())
        ].copy()
        
        if valid_test.empty:
            return None, 0

        eval_user_count = valid_test['UserID'].nunique()
        valid_test['u_idx'] = valid_test['UserID'].map(user_map)
        valid_test['i_idx'] = valid_test['MovieID'].map(item_map)
        
        user_ground_truth = valid_test.groupby('u_idx')['i_idx'].apply(set).to_dict()
        test_users = list(user_ground_truth.keys())
        
        u_emb_test = final_user_emb[test_users]
        scores = torch.matmul(u_emb_test, final_item_emb.t())
        
        idx_to_movie = {v: k for k, v in item_map.items()}
        idx_to_user  = {v: k for k, v in user_map.items()}

        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {
            'mainstream': {'high': 0, 'medium': 0, 'low': 0},
            'niche': {'high': 0, 'medium': 0, 'low': 0}
        }
        total_recs_per_group = {'mainstream': 0, 'niche': 0}
        item_rec_counts = np.zeros(len(item_map))
        
        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_list = []
        ndcg_list = []

        for i, u_idx in enumerate(test_users):
            recs = top_indices[i]
            truth = user_ground_truth[u_idx]
            
            hits = len(set(recs) & truth)
            rec = hits / len(truth)
            recall_list.append(rec)
            
            dcg, idcg = 0.0, 0.0
            for k, item in enumerate(recs):
                if item in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_list.append(dcg / idcg if idcg > 0 else 0)
            
            u_id = idx_to_user[u_idx]
            u_type = user_labels.get(u_id, 'niche') 
            group_recalls[u_type].append(rec)
            total_recs_per_group[u_type] += K
            
            for item_idx in recs:
                item_rec_counts[item_idx] += 1
                tier = movie_tiers.get(idx_to_movie[item_idx], 'low')
                pop_counts[u_type][tier] += 1

        avg_recall = np.mean(recall_list)
        avg_ndcg = np.mean(ndcg_list)
        eo_score = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] and group_recalls['niche'] else 0
        
        total_recs = total_recs_per_group['mainstream'] + total_recs_per_group['niche']
        dp_score = abs(total_recs_per_group['mainstream']/total_recs - total_recs_per_group['niche']/total_recs) if total_recs > 0 else 0

        gini_score = get_gini(item_rec_counts)
        
        formatted_pop = {grp: {t: f"{(pop_counts[grp][t] / total_recs_per_group[grp] * 100):.2f}%" if total_recs_per_group[grp] > 0 else "0.00%" for t in ['low', 'medium', 'high']} for grp in ['mainstream', 'niche']}
                    
        return {
            'Recall': avg_recall, 'NDCG': avg_ndcg, 'EO': eo_score,
            'DP': dp_score, 'Gini': gini_score, 'PopExposure': formatted_pop
        }, eval_user_count

# ----------------------------
# 2. Baseline Model (No Graph Building)
# ----------------------------
class BaselineModel(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super(BaselineModel, self).__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        self.transform = nn.Linear(embedding_dim, embedding_dim)
        
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        
    def forward(self):
        # [MODIFIED]: No inputs required. Simply returns transformed embeddings.
        u_final = F.relu(self.transform(self.user_emb.weight))
        i_final = F.relu(self.item_emb.weight) # Removed transform from items to match standard BCF
        
        return u_final, i_final

# ----------------------------
# 3. Execution Loop
# ----------------------------
print(f"Recommending 10 movies using Streamlined Baseline")
LR, EPOCHS, BATCH_SIZE, REG_LAMBDA = 0.001, 50, 2048, 1e-4

slice_nums = sorted(ratings['SliceNum'].unique())
print(f"{'Slice Pair':<15} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 88)

for s in slice_nums[:-1]:
    train_s, test_s = s, s + 1
    train_df = ratings[ratings['SliceNum'] == train_s].copy()
    test_df  = ratings[ratings['SliceNum'] == test_s].copy()
    
    try:
        user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{test_s}.csv").set_index('UserID')['UserType'].to_dict()
        movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{test_s}.csv").set_index('MovieID')['PopTier'].to_dict()
    except: user_labels, movie_tiers = {}, {}

    unique_users, unique_items = train_df['UserID'].unique(), train_df['MovieID'].unique()
    user_map = {uid: i for i, uid in enumerate(unique_users)}
    item_map = {iid: i for i, iid in enumerate(unique_items)}
    num_users, num_items = len(user_map), len(item_map)
    
    train_u = train_df['UserID'].map(user_map).values
    train_i = train_df['MovieID'].map(item_map).values
    
    # [MODIFIED]: build_kg_adj REMOVED
    
    model = BaselineModel(num_users, num_items).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        
        indices = np.arange(len(train_u))
        np.random.shuffle(indices)
        
        for i in range(0, len(indices), BATCH_SIZE):
            # Compute embeddings
            u_final, i_final = model()

            batch_idx = indices[i:i+BATCH_SIZE]
            u_b = torch.LongTensor(train_u[batch_idx]).to(device)
            pos_b = torch.LongTensor(train_i[batch_idx]).to(device)
            neg_b = torch.randint(0, num_items, (len(batch_idx),)).to(device)
            
            pos_scores = (u_final[u_b] * i_final[pos_b]).sum(dim=1)
            neg_scores = (u_final[u_b] * i_final[neg_b]).sum(dim=1)
            
            bpr_loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
            
            reg_loss = REG_LAMBDA * (model.user_emb.weight[u_b].norm(2).pow(2) + 
                                     model.item_emb.weight[pos_b].norm(2).pow(2)) / len(batch_idx)
            
            loss = bpr_loss + reg_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # [MODIFIED]: compute_metrics no longer takes adj_info
    metrics, user_count = compute_metrics(model, test_df, train_df, user_map, item_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{train_s} -> {test_s:<8} | {user_count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"Popularity exposure (percent): {metrics['PopExposure']}")
        print("-" * 88)

Using device: cuda
Recommending 10 movies using Streamlined Baseline
Slice Pair      | Users  | Recall   | NDCG     | EO       | DP       | Gini    
----------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.0125    | 0.0672    | 0.0005    | 0.1875    | 0.9920
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.26%', 'high': '99.74%'}, 'niche': {'low': '0.00%', 'medium': '0.88%', 'high': '99.12%'}}
----------------------------------------------------------------------------------------
2 -> 3        | 117    | 0.0140    | 0.0635    | 0.0193    | 0.2137    | 0.9931
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.00%', 'high': '100.00%'}, 'niche': {'low': '0.00%', 'medium': '0.14%', 'high': '99.86%'}}
----------------------------------------------------------------------------------------
3 -> 4        | 166    | 0.0262    | 0.0540    | 0.0443    | 0.3855    | 0.9905
Popularity ex

K = 25

In [6]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import scipy.sparse as sp
import os

# ----------------------------
# 0. Deterministic Seeding
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ----------------------------
# 1. Metric Helper Functions
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.array(recommendation_counts)
    if np.sum(arr) == 0: return 0.0
    arr = np.sort(arr)
    n = len(arr)
    cumvals = np.cumsum(arr)
    gini = (n + 1 - 2 * np.sum(cumvals) / cumvals[-1]) / n
    return gini

def compute_metrics(model, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=25):
    model.eval()
    with torch.no_grad():
        # [MODIFIED]: Removed adj_matrix dependency
        final_user_emb, final_item_emb = model()
        
        valid_test = test_df[
            test_df['UserID'].isin(user_map.keys()) & 
            test_df['MovieID'].isin(item_map.keys())
        ].copy()
        
        if valid_test.empty:
            return None, 0

        eval_user_count = valid_test['UserID'].nunique()
        valid_test['u_idx'] = valid_test['UserID'].map(user_map)
        valid_test['i_idx'] = valid_test['MovieID'].map(item_map)
        
        user_ground_truth = valid_test.groupby('u_idx')['i_idx'].apply(set).to_dict()
        test_users = list(user_ground_truth.keys())
        
        u_emb_test = final_user_emb[test_users]
        scores = torch.matmul(u_emb_test, final_item_emb.t())
        
        idx_to_movie = {v: k for k, v in item_map.items()}
        idx_to_user  = {v: k for k, v in user_map.items()}

        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {
            'mainstream': {'high': 0, 'medium': 0, 'low': 0},
            'niche': {'high': 0, 'medium': 0, 'low': 0}
        }
        total_recs_per_group = {'mainstream': 0, 'niche': 0}
        item_rec_counts = np.zeros(len(item_map))
        
        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_list = []
        ndcg_list = []

        for i, u_idx in enumerate(test_users):
            recs = top_indices[i]
            truth = user_ground_truth[u_idx]
            
            hits = len(set(recs) & truth)
            rec = hits / len(truth)
            recall_list.append(rec)
            
            dcg, idcg = 0.0, 0.0
            for k, item in enumerate(recs):
                if item in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_list.append(dcg / idcg if idcg > 0 else 0)
            
            u_id = idx_to_user[u_idx]
            u_type = user_labels.get(u_id, 'niche') 
            group_recalls[u_type].append(rec)
            total_recs_per_group[u_type] += K
            
            for item_idx in recs:
                item_rec_counts[item_idx] += 1
                tier = movie_tiers.get(idx_to_movie[item_idx], 'low')
                pop_counts[u_type][tier] += 1

        avg_recall = np.mean(recall_list)
        avg_ndcg = np.mean(ndcg_list)
        eo_score = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] and group_recalls['niche'] else 0
        
        total_recs = total_recs_per_group['mainstream'] + total_recs_per_group['niche']
        dp_score = abs(total_recs_per_group['mainstream']/total_recs - total_recs_per_group['niche']/total_recs) if total_recs > 0 else 0

        gini_score = get_gini(item_rec_counts)
        
        formatted_pop = {grp: {t: f"{(pop_counts[grp][t] / total_recs_per_group[grp] * 100):.2f}%" if total_recs_per_group[grp] > 0 else "0.00%" for t in ['low', 'medium', 'high']} for grp in ['mainstream', 'niche']}
                    
        return {
            'Recall': avg_recall, 'NDCG': avg_ndcg, 'EO': eo_score,
            'DP': dp_score, 'Gini': gini_score, 'PopExposure': formatted_pop
        }, eval_user_count

# ----------------------------
# 2. Baseline Model (No Graph Building)
# ----------------------------
class BaselineModel(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super(BaselineModel, self).__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        self.transform = nn.Linear(embedding_dim, embedding_dim)
        
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        
    def forward(self):
        # [MODIFIED]: No inputs required. Simply returns transformed embeddings.
        u_final = F.relu(self.transform(self.user_emb.weight))
        i_final = F.relu(self.item_emb.weight) # Removed transform from items to match standard BCF
        
        return u_final, i_final

# ----------------------------
# 3. Execution Loop
# ----------------------------
print(f"Recommending 25 movies using Streamlined Baseline")
LR, EPOCHS, BATCH_SIZE, REG_LAMBDA = 0.001, 50, 2048, 1e-4

slice_nums = sorted(ratings['SliceNum'].unique())
print(f"{'Slice Pair':<15} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 88)

for s in slice_nums[:-1]:
    train_s, test_s = s, s + 1
    train_df = ratings[ratings['SliceNum'] == train_s].copy()
    test_df  = ratings[ratings['SliceNum'] == test_s].copy()
    
    try:
        user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{test_s}.csv").set_index('UserID')['UserType'].to_dict()
        movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{test_s}.csv").set_index('MovieID')['PopTier'].to_dict()
    except: user_labels, movie_tiers = {}, {}

    unique_users, unique_items = train_df['UserID'].unique(), train_df['MovieID'].unique()
    user_map = {uid: i for i, uid in enumerate(unique_users)}
    item_map = {iid: i for i, iid in enumerate(unique_items)}
    num_users, num_items = len(user_map), len(item_map)
    
    train_u = train_df['UserID'].map(user_map).values
    train_i = train_df['MovieID'].map(item_map).values
    
    # [MODIFIED]: build_kg_adj REMOVED
    
    model = BaselineModel(num_users, num_items).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        
        indices = np.arange(len(train_u))
        np.random.shuffle(indices)
        
        for i in range(0, len(indices), BATCH_SIZE):
            # Compute embeddings
            u_final, i_final = model()

            batch_idx = indices[i:i+BATCH_SIZE]
            u_b = torch.LongTensor(train_u[batch_idx]).to(device)
            pos_b = torch.LongTensor(train_i[batch_idx]).to(device)
            neg_b = torch.randint(0, num_items, (len(batch_idx),)).to(device)
            
            pos_scores = (u_final[u_b] * i_final[pos_b]).sum(dim=1)
            neg_scores = (u_final[u_b] * i_final[neg_b]).sum(dim=1)
            
            bpr_loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
            
            reg_loss = REG_LAMBDA * (model.user_emb.weight[u_b].norm(2).pow(2) + 
                                     model.item_emb.weight[pos_b].norm(2).pow(2)) / len(batch_idx)
            
            loss = bpr_loss + reg_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # [MODIFIED]: compute_metrics no longer takes adj_info
    metrics, user_count = compute_metrics(model, test_df, train_df, user_map, item_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{train_s} -> {test_s:<8} | {user_count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"Popularity exposure (percent): {metrics['PopExposure']}")
        print("-" * 88)

Using device: cuda
Recommending 25 movies using Streamlined Baseline
Slice Pair      | Users  | Recall   | NDCG     | EO       | DP       | Gini    
----------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.0421    | 0.0709    | 0.0016    | 0.1875    | 0.9846
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.21%', 'high': '99.79%'}, 'niche': {'low': '0.00%', 'medium': '0.84%', 'high': '99.16%'}}
----------------------------------------------------------------------------------------
2 -> 3        | 117    | 0.0385    | 0.0704    | 0.0344    | 0.2137    | 0.9851
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.17%', 'high': '99.83%'}, 'niche': {'low': '0.00%', 'medium': '0.28%', 'high': '99.72%'}}
----------------------------------------------------------------------------------------
3 -> 4        | 166    | 0.0488    | 0.0615    | 0.0643    | 0.3855    | 0.9809
Popularity exp

K = 50

In [7]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import scipy.sparse as sp
import os

# ----------------------------
# 0. Deterministic Seeding
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ----------------------------
# 1. Metric Helper Functions
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.array(recommendation_counts)
    if np.sum(arr) == 0: return 0.0
    arr = np.sort(arr)
    n = len(arr)
    cumvals = np.cumsum(arr)
    gini = (n + 1 - 2 * np.sum(cumvals) / cumvals[-1]) / n
    return gini

def compute_metrics(model, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=50):
    model.eval()
    with torch.no_grad():
        # [MODIFIED]: Removed adj_matrix dependency
        final_user_emb, final_item_emb = model()
        
        valid_test = test_df[
            test_df['UserID'].isin(user_map.keys()) & 
            test_df['MovieID'].isin(item_map.keys())
        ].copy()
        
        if valid_test.empty:
            return None, 0

        eval_user_count = valid_test['UserID'].nunique()
        valid_test['u_idx'] = valid_test['UserID'].map(user_map)
        valid_test['i_idx'] = valid_test['MovieID'].map(item_map)
        
        user_ground_truth = valid_test.groupby('u_idx')['i_idx'].apply(set).to_dict()
        test_users = list(user_ground_truth.keys())
        
        u_emb_test = final_user_emb[test_users]
        scores = torch.matmul(u_emb_test, final_item_emb.t())
        
        idx_to_movie = {v: k for k, v in item_map.items()}
        idx_to_user  = {v: k for k, v in user_map.items()}

        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {
            'mainstream': {'high': 0, 'medium': 0, 'low': 0},
            'niche': {'high': 0, 'medium': 0, 'low': 0}
        }
        total_recs_per_group = {'mainstream': 0, 'niche': 0}
        item_rec_counts = np.zeros(len(item_map))
        
        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_list = []
        ndcg_list = []

        for i, u_idx in enumerate(test_users):
            recs = top_indices[i]
            truth = user_ground_truth[u_idx]
            
            hits = len(set(recs) & truth)
            rec = hits / len(truth)
            recall_list.append(rec)
            
            dcg, idcg = 0.0, 0.0
            for k, item in enumerate(recs):
                if item in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_list.append(dcg / idcg if idcg > 0 else 0)
            
            u_id = idx_to_user[u_idx]
            u_type = user_labels.get(u_id, 'niche') 
            group_recalls[u_type].append(rec)
            total_recs_per_group[u_type] += K
            
            for item_idx in recs:
                item_rec_counts[item_idx] += 1
                tier = movie_tiers.get(idx_to_movie[item_idx], 'low')
                pop_counts[u_type][tier] += 1

        avg_recall = np.mean(recall_list)
        avg_ndcg = np.mean(ndcg_list)
        eo_score = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] and group_recalls['niche'] else 0
        
        total_recs = total_recs_per_group['mainstream'] + total_recs_per_group['niche']
        dp_score = abs(total_recs_per_group['mainstream']/total_recs - total_recs_per_group['niche']/total_recs) if total_recs > 0 else 0

        gini_score = get_gini(item_rec_counts)
        
        formatted_pop = {grp: {t: f"{(pop_counts[grp][t] / total_recs_per_group[grp] * 100):.2f}%" if total_recs_per_group[grp] > 0 else "0.00%" for t in ['low', 'medium', 'high']} for grp in ['mainstream', 'niche']}
                    
        return {
            'Recall': avg_recall, 'NDCG': avg_ndcg, 'EO': eo_score,
            'DP': dp_score, 'Gini': gini_score, 'PopExposure': formatted_pop
        }, eval_user_count

# ----------------------------
# 2. Baseline Model (No Graph Building)
# ----------------------------
class BaselineModel(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super(BaselineModel, self).__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        self.transform = nn.Linear(embedding_dim, embedding_dim)
        
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        
    def forward(self):
        # [MODIFIED]: No inputs required. Simply returns transformed embeddings.
        u_final = F.relu(self.transform(self.user_emb.weight))
        i_final = F.relu(self.item_emb.weight) # Removed transform from items to match standard BCF
        
        return u_final, i_final

# ----------------------------
# 3. Execution Loop
# ----------------------------
print(f"Recommending 50 movies using Streamlined Baseline")
LR, EPOCHS, BATCH_SIZE, REG_LAMBDA = 0.001, 50, 2048, 1e-4

slice_nums = sorted(ratings['SliceNum'].unique())
print(f"{'Slice Pair':<15} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 88)

for s in slice_nums[:-1]:
    train_s, test_s = s, s + 1
    train_df = ratings[ratings['SliceNum'] == train_s].copy()
    test_df  = ratings[ratings['SliceNum'] == test_s].copy()
    
    try:
        user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{test_s}.csv").set_index('UserID')['UserType'].to_dict()
        movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{test_s}.csv").set_index('MovieID')['PopTier'].to_dict()
    except: user_labels, movie_tiers = {}, {}

    unique_users, unique_items = train_df['UserID'].unique(), train_df['MovieID'].unique()
    user_map = {uid: i for i, uid in enumerate(unique_users)}
    item_map = {iid: i for i, iid in enumerate(unique_items)}
    num_users, num_items = len(user_map), len(item_map)
    
    train_u = train_df['UserID'].map(user_map).values
    train_i = train_df['MovieID'].map(item_map).values
    
    # [MODIFIED]: build_kg_adj REMOVED
    
    model = BaselineModel(num_users, num_items).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        
        indices = np.arange(len(train_u))
        np.random.shuffle(indices)
        
        for i in range(0, len(indices), BATCH_SIZE):
            # Compute embeddings
            u_final, i_final = model()

            batch_idx = indices[i:i+BATCH_SIZE]
            u_b = torch.LongTensor(train_u[batch_idx]).to(device)
            pos_b = torch.LongTensor(train_i[batch_idx]).to(device)
            neg_b = torch.randint(0, num_items, (len(batch_idx),)).to(device)
            
            pos_scores = (u_final[u_b] * i_final[pos_b]).sum(dim=1)
            neg_scores = (u_final[u_b] * i_final[neg_b]).sum(dim=1)
            
            bpr_loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
            
            reg_loss = REG_LAMBDA * (model.user_emb.weight[u_b].norm(2).pow(2) + 
                                     model.item_emb.weight[pos_b].norm(2).pow(2)) / len(batch_idx)
            
            loss = bpr_loss + reg_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # [MODIFIED]: compute_metrics no longer takes adj_info
    metrics, user_count = compute_metrics(model, test_df, train_df, user_map, item_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{train_s} -> {test_s:<8} | {user_count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"Popularity exposure (percent): {metrics['PopExposure']}")
        print("-" * 88)

Using device: cuda
Recommending 50 movies using Streamlined Baseline
Slice Pair      | Users  | Recall   | NDCG     | EO       | DP       | Gini    
----------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.0852    | 0.0845    | 0.0361    | 0.1875    | 0.9705
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.31%', 'high': '99.69%'}, 'niche': {'low': '0.07%', 'medium': '1.37%', 'high': '98.56%'}}
----------------------------------------------------------------------------------------
2 -> 3        | 117    | 0.0636    | 0.0746    | 0.0514    | 0.2137    | 0.9734
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.13%', 'high': '99.87%'}, 'niche': {'low': '0.00%', 'medium': '0.45%', 'high': '99.55%'}}
----------------------------------------------------------------------------------------
3 -> 4        | 166    | 0.0770    | 0.0712    | 0.0823    | 0.3855    | 0.9673
Popularity exp

K = 75

In [8]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import scipy.sparse as sp
import os

# ----------------------------
# 0. Deterministic Seeding
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ----------------------------
# 1. Metric Helper Functions
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.array(recommendation_counts)
    if np.sum(arr) == 0: return 0.0
    arr = np.sort(arr)
    n = len(arr)
    cumvals = np.cumsum(arr)
    gini = (n + 1 - 2 * np.sum(cumvals) / cumvals[-1]) / n
    return gini

def compute_metrics(model, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=75):
    model.eval()
    with torch.no_grad():
        # [MODIFIED]: Removed adj_matrix dependency
        final_user_emb, final_item_emb = model()
        
        valid_test = test_df[
            test_df['UserID'].isin(user_map.keys()) & 
            test_df['MovieID'].isin(item_map.keys())
        ].copy()
        
        if valid_test.empty:
            return None, 0

        eval_user_count = valid_test['UserID'].nunique()
        valid_test['u_idx'] = valid_test['UserID'].map(user_map)
        valid_test['i_idx'] = valid_test['MovieID'].map(item_map)
        
        user_ground_truth = valid_test.groupby('u_idx')['i_idx'].apply(set).to_dict()
        test_users = list(user_ground_truth.keys())
        
        u_emb_test = final_user_emb[test_users]
        scores = torch.matmul(u_emb_test, final_item_emb.t())
        
        idx_to_movie = {v: k for k, v in item_map.items()}
        idx_to_user  = {v: k for k, v in user_map.items()}

        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {
            'mainstream': {'high': 0, 'medium': 0, 'low': 0},
            'niche': {'high': 0, 'medium': 0, 'low': 0}
        }
        total_recs_per_group = {'mainstream': 0, 'niche': 0}
        item_rec_counts = np.zeros(len(item_map))
        
        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_list = []
        ndcg_list = []

        for i, u_idx in enumerate(test_users):
            recs = top_indices[i]
            truth = user_ground_truth[u_idx]
            
            hits = len(set(recs) & truth)
            rec = hits / len(truth)
            recall_list.append(rec)
            
            dcg, idcg = 0.0, 0.0
            for k, item in enumerate(recs):
                if item in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_list.append(dcg / idcg if idcg > 0 else 0)
            
            u_id = idx_to_user[u_idx]
            u_type = user_labels.get(u_id, 'niche') 
            group_recalls[u_type].append(rec)
            total_recs_per_group[u_type] += K
            
            for item_idx in recs:
                item_rec_counts[item_idx] += 1
                tier = movie_tiers.get(idx_to_movie[item_idx], 'low')
                pop_counts[u_type][tier] += 1

        avg_recall = np.mean(recall_list)
        avg_ndcg = np.mean(ndcg_list)
        eo_score = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] and group_recalls['niche'] else 0
        
        total_recs = total_recs_per_group['mainstream'] + total_recs_per_group['niche']
        dp_score = abs(total_recs_per_group['mainstream']/total_recs - total_recs_per_group['niche']/total_recs) if total_recs > 0 else 0

        gini_score = get_gini(item_rec_counts)
        
        formatted_pop = {grp: {t: f"{(pop_counts[grp][t] / total_recs_per_group[grp] * 100):.2f}%" if total_recs_per_group[grp] > 0 else "0.00%" for t in ['low', 'medium', 'high']} for grp in ['mainstream', 'niche']}
                    
        return {
            'Recall': avg_recall, 'NDCG': avg_ndcg, 'EO': eo_score,
            'DP': dp_score, 'Gini': gini_score, 'PopExposure': formatted_pop
        }, eval_user_count

# ----------------------------
# 2. Baseline Model (No Graph Building)
# ----------------------------
class BaselineModel(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super(BaselineModel, self).__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        self.transform = nn.Linear(embedding_dim, embedding_dim)
        
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        
    def forward(self):
        # [MODIFIED]: No inputs required. Simply returns transformed embeddings.
        u_final = F.relu(self.transform(self.user_emb.weight))
        i_final = F.relu(self.item_emb.weight) # Removed transform from items to match standard BCF
        
        return u_final, i_final

# ----------------------------
# 3. Execution Loop
# ----------------------------
print(f"Recommending 75 movies using Streamlined Baseline")
LR, EPOCHS, BATCH_SIZE, REG_LAMBDA = 0.001, 50, 2048, 1e-4

slice_nums = sorted(ratings['SliceNum'].unique())
print(f"{'Slice Pair':<15} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 88)

for s in slice_nums[:-1]:
    train_s, test_s = s, s + 1
    train_df = ratings[ratings['SliceNum'] == train_s].copy()
    test_df  = ratings[ratings['SliceNum'] == test_s].copy()
    
    try:
        user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{test_s}.csv").set_index('UserID')['UserType'].to_dict()
        movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{test_s}.csv").set_index('MovieID')['PopTier'].to_dict()
    except: user_labels, movie_tiers = {}, {}

    unique_users, unique_items = train_df['UserID'].unique(), train_df['MovieID'].unique()
    user_map = {uid: i for i, uid in enumerate(unique_users)}
    item_map = {iid: i for i, iid in enumerate(unique_items)}
    num_users, num_items = len(user_map), len(item_map)
    
    train_u = train_df['UserID'].map(user_map).values
    train_i = train_df['MovieID'].map(item_map).values
    
    # [MODIFIED]: build_kg_adj REMOVED
    
    model = BaselineModel(num_users, num_items).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        
        indices = np.arange(len(train_u))
        np.random.shuffle(indices)
        
        for i in range(0, len(indices), BATCH_SIZE):
            # Compute embeddings
            u_final, i_final = model()

            batch_idx = indices[i:i+BATCH_SIZE]
            u_b = torch.LongTensor(train_u[batch_idx]).to(device)
            pos_b = torch.LongTensor(train_i[batch_idx]).to(device)
            neg_b = torch.randint(0, num_items, (len(batch_idx),)).to(device)
            
            pos_scores = (u_final[u_b] * i_final[pos_b]).sum(dim=1)
            neg_scores = (u_final[u_b] * i_final[neg_b]).sum(dim=1)
            
            bpr_loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
            
            reg_loss = REG_LAMBDA * (model.user_emb.weight[u_b].norm(2).pow(2) + 
                                     model.item_emb.weight[pos_b].norm(2).pow(2)) / len(batch_idx)
            
            loss = bpr_loss + reg_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # [MODIFIED]: compute_metrics no longer takes adj_info
    metrics, user_count = compute_metrics(model, test_df, train_df, user_map, item_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{train_s} -> {test_s:<8} | {user_count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"Popularity exposure (percent): {metrics['PopExposure']}")
        print("-" * 88)

Using device: cuda
Recommending 75 movies using Streamlined Baseline
Slice Pair      | Users  | Recall   | NDCG     | EO       | DP       | Gini    
----------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.1283    | 0.0965    | 0.0964    | 0.1875    | 0.9581
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.65%', 'high': '99.35%'}, 'niche': {'low': '0.09%', 'medium': '1.64%', 'high': '98.27%'}}
----------------------------------------------------------------------------------------
2 -> 3        | 117    | 0.0914    | 0.0823    | 0.0899    | 0.2137    | 0.9621
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.14%', 'high': '99.86%'}, 'niche': {'low': '0.00%', 'medium': '0.86%', 'high': '99.14%'}}
----------------------------------------------------------------------------------------
3 -> 4        | 166    | 0.1195    | 0.0839    | 0.0852    | 0.3855    | 0.9559
Popularity exp

K = 100

In [9]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import scipy.sparse as sp
import os

# ----------------------------
# 0. Deterministic Seeding
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ----------------------------
# 1. Metric Helper Functions
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.array(recommendation_counts)
    if np.sum(arr) == 0: return 0.0
    arr = np.sort(arr)
    n = len(arr)
    cumvals = np.cumsum(arr)
    gini = (n + 1 - 2 * np.sum(cumvals) / cumvals[-1]) / n
    return gini

def compute_metrics(model, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=100):
    model.eval()
    with torch.no_grad():
        # [MODIFIED]: Removed adj_matrix dependency
        final_user_emb, final_item_emb = model()
        
        valid_test = test_df[
            test_df['UserID'].isin(user_map.keys()) & 
            test_df['MovieID'].isin(item_map.keys())
        ].copy()
        
        if valid_test.empty:
            return None, 0

        eval_user_count = valid_test['UserID'].nunique()
        valid_test['u_idx'] = valid_test['UserID'].map(user_map)
        valid_test['i_idx'] = valid_test['MovieID'].map(item_map)
        
        user_ground_truth = valid_test.groupby('u_idx')['i_idx'].apply(set).to_dict()
        test_users = list(user_ground_truth.keys())
        
        u_emb_test = final_user_emb[test_users]
        scores = torch.matmul(u_emb_test, final_item_emb.t())
        
        idx_to_movie = {v: k for k, v in item_map.items()}
        idx_to_user  = {v: k for k, v in user_map.items()}

        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {
            'mainstream': {'high': 0, 'medium': 0, 'low': 0},
            'niche': {'high': 0, 'medium': 0, 'low': 0}
        }
        total_recs_per_group = {'mainstream': 0, 'niche': 0}
        item_rec_counts = np.zeros(len(item_map))
        
        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_list = []
        ndcg_list = []

        for i, u_idx in enumerate(test_users):
            recs = top_indices[i]
            truth = user_ground_truth[u_idx]
            
            hits = len(set(recs) & truth)
            rec = hits / len(truth)
            recall_list.append(rec)
            
            dcg, idcg = 0.0, 0.0
            for k, item in enumerate(recs):
                if item in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_list.append(dcg / idcg if idcg > 0 else 0)
            
            u_id = idx_to_user[u_idx]
            u_type = user_labels.get(u_id, 'niche') 
            group_recalls[u_type].append(rec)
            total_recs_per_group[u_type] += K
            
            for item_idx in recs:
                item_rec_counts[item_idx] += 1
                tier = movie_tiers.get(idx_to_movie[item_idx], 'low')
                pop_counts[u_type][tier] += 1

        avg_recall = np.mean(recall_list)
        avg_ndcg = np.mean(ndcg_list)
        eo_score = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] and group_recalls['niche'] else 0
        
        total_recs = total_recs_per_group['mainstream'] + total_recs_per_group['niche']
        dp_score = abs(total_recs_per_group['mainstream']/total_recs - total_recs_per_group['niche']/total_recs) if total_recs > 0 else 0

        gini_score = get_gini(item_rec_counts)
        
        formatted_pop = {grp: {t: f"{(pop_counts[grp][t] / total_recs_per_group[grp] * 100):.2f}%" if total_recs_per_group[grp] > 0 else "0.00%" for t in ['low', 'medium', 'high']} for grp in ['mainstream', 'niche']}
                    
        return {
            'Recall': avg_recall, 'NDCG': avg_ndcg, 'EO': eo_score,
            'DP': dp_score, 'Gini': gini_score, 'PopExposure': formatted_pop
        }, eval_user_count

# ----------------------------
# 2. Baseline Model (No Graph Building)
# ----------------------------
class BaselineModel(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super(BaselineModel, self).__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        self.transform = nn.Linear(embedding_dim, embedding_dim)
        
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        
    def forward(self):
        # [MODIFIED]: No inputs required. Simply returns transformed embeddings.
        u_final = F.relu(self.transform(self.user_emb.weight))
        i_final = F.relu(self.item_emb.weight) # Removed transform from items to match standard BCF
        
        return u_final, i_final

# ----------------------------
# 3. Execution Loop
# ----------------------------
print(f"Recommending 100 movies using Streamlined Baseline")
LR, EPOCHS, BATCH_SIZE, REG_LAMBDA = 0.001, 50, 2048, 1e-4

slice_nums = sorted(ratings['SliceNum'].unique())
print(f"{'Slice Pair':<15} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 88)

for s in slice_nums[:-1]:
    train_s, test_s = s, s + 1
    train_df = ratings[ratings['SliceNum'] == train_s].copy()
    test_df  = ratings[ratings['SliceNum'] == test_s].copy()
    
    try:
        user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{test_s}.csv").set_index('UserID')['UserType'].to_dict()
        movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{test_s}.csv").set_index('MovieID')['PopTier'].to_dict()
    except: user_labels, movie_tiers = {}, {}

    unique_users, unique_items = train_df['UserID'].unique(), train_df['MovieID'].unique()
    user_map = {uid: i for i, uid in enumerate(unique_users)}
    item_map = {iid: i for i, iid in enumerate(unique_items)}
    num_users, num_items = len(user_map), len(item_map)
    
    train_u = train_df['UserID'].map(user_map).values
    train_i = train_df['MovieID'].map(item_map).values
    
    # [MODIFIED]: build_kg_adj REMOVED
    
    model = BaselineModel(num_users, num_items).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        
        indices = np.arange(len(train_u))
        np.random.shuffle(indices)
        
        for i in range(0, len(indices), BATCH_SIZE):
            # Compute embeddings
            u_final, i_final = model()

            batch_idx = indices[i:i+BATCH_SIZE]
            u_b = torch.LongTensor(train_u[batch_idx]).to(device)
            pos_b = torch.LongTensor(train_i[batch_idx]).to(device)
            neg_b = torch.randint(0, num_items, (len(batch_idx),)).to(device)
            
            pos_scores = (u_final[u_b] * i_final[pos_b]).sum(dim=1)
            neg_scores = (u_final[u_b] * i_final[neg_b]).sum(dim=1)
            
            bpr_loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
            
            reg_loss = REG_LAMBDA * (model.user_emb.weight[u_b].norm(2).pow(2) + 
                                     model.item_emb.weight[pos_b].norm(2).pow(2)) / len(batch_idx)
            
            loss = bpr_loss + reg_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # [MODIFIED]: compute_metrics no longer takes adj_info
    metrics, user_count = compute_metrics(model, test_df, train_df, user_map, item_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{train_s} -> {test_s:<8} | {user_count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"Popularity exposure (percent): {metrics['PopExposure']}")
        print("-" * 88)

Using device: cuda
Recommending 100 movies using Streamlined Baseline
Slice Pair      | Users  | Recall   | NDCG     | EO       | DP       | Gini    
----------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.1539    | 0.1065    | 0.1194    | 0.1875    | 0.9472
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.92%', 'high': '99.08%'}, 'niche': {'low': '0.12%', 'medium': '2.00%', 'high': '97.88%'}}
----------------------------------------------------------------------------------------
2 -> 3        | 117    | 0.1434    | 0.0974    | 0.1750    | 0.2137    | 0.9527
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.22%', 'high': '99.78%'}, 'niche': {'low': '0.00%', 'medium': '1.18%', 'high': '98.82%'}}
----------------------------------------------------------------------------------------
3 -> 4        | 166    | 0.1564    | 0.0953    | 0.1102    | 0.3855    | 0.9450
Popularity ex